# BÀI THỰC HÀNH 2 — Data Understanding, EDA & Preprocessing

**Phiên bản:** STUDENT TODO

**Dataset:** UCI Student Performance — `student-por.csv`

- Trang UCI: https://archive.ics.uci.edu/dataset/320/student%2Bperformance
- ZIP trực tiếp: https://archive.ics.uci.edu/static/public/320/student%2Bperformance.zip
- Citation: Cortez, P. (2008). *Student Performance* [Dataset]. UCI Machine Learning Repository. DOI: 10.24432/C5TG7T.

> **Lưu ý:** Nếu chạy trên Google Colab/Jupyter có Internet, notebook có thể tự tải ZIP chính thức. Nếu môi trường không có Internet, hãy tải `student.zip`, giải nén và đặt `student-por.csv` cùng thư mục notebook.

In [ ]:
# Cell thiết lập chung - được phép chạy nguyên trạng
from pathlib import Path
import io, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_FILE = Path('student-por.csv')
UCI_ZIP_URL = 'https://archive.ics.uci.edu/static/public/320/student%2Bperformance.zip'

def ensure_dataset():
    if DATA_FILE.exists():
        print(f'Found local dataset: {DATA_FILE.resolve()}')
        return DATA_FILE
    try:
        import requests
        r = requests.get(UCI_ZIP_URL, timeout=30)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            member = next(n for n in z.namelist() if n.endswith('student-por.csv'))
            DATA_FILE.write_bytes(z.read(member))
        print(f'Downloaded dataset to: {DATA_FILE.resolve()}')
        return DATA_FILE
    except Exception as e:
        raise FileNotFoundError(
            'Không tìm thấy student-por.csv và không thể tự tải dữ liệu. '
            'Hãy tải student.zip từ UCI, giải nén và đặt student-por.csv cùng thư mục notebook.'
        ) from e

DATA_PATH = ensure_dataset()


## 1. Đọc dữ liệu và hiểu cấu trúc
**Yêu cầu:** đọc CSV với `sep=";"`; in shape, `head()`, `info()`, danh sách cột. Trả lời: một dòng đại diện cho gì? biến nào là điểm cuối cùng?

In [ ]:
# TODO 1: Đọc dữ liệu vào biến df
df = pd.read_csv(DATA_PATH, sep=';')

# in shape, head(), info(), columns
print("Shape của dữ liệu:", df.shape)
display(df.head())
df.info()
print("Danh sách các cột:", df.columns.tolist())


### Câu trả lời ngắn
- Số dòng: **649**
- Số cột: **33**
- Unit of analysis: **Một học sinh (student)**
- Biến điểm cuối cùng: **G3**

## 2. Phân loại kiểu dữ liệu
Tạo một bảng Data Dictionary rút gọn cho ít nhất 15 biến với các cột: `variable`, `meaning`, `storage_dtype`, `semantic_type` (nominal/ordinal/binary/discrete numerical/numerical).

In [ ]:
# TODO 2: Tạo Data Dictionary rút gọn
data_dict = pd.DataFrame([
    {'variable': 'school', 'meaning': 'Trường học', 'storage_dtype': 'object', 'semantic_type': 'nominal'},
    {'variable': 'sex', 'meaning': 'Giới tính', 'storage_dtype': 'object', 'semantic_type': 'binary'},
    {'variable': 'age', 'meaning': 'Tuổi', 'storage_dtype': 'int64', 'semantic_type': 'discrete numerical'},
    {'variable': 'address', 'meaning': 'Khu vực sống', 'storage_dtype': 'object', 'semantic_type': 'binary'},
    {'variable': 'famsize', 'meaning': 'Quy mô gia đình', 'storage_dtype': 'object', 'semantic_type': 'binary'},
    {'variable': 'Pstatus', 'meaning': 'Tình trạng chung sống của cha mẹ', 'storage_dtype': 'object', 'semantic_type': 'binary'},
    {'variable': 'Medu', 'meaning': 'Trình độ học vấn của mẹ', 'storage_dtype': 'int64', 'semantic_type': 'ordinal'},
    {'variable': 'Fedu', 'meaning': 'Trình độ học vấn của cha', 'storage_dtype': 'int64', 'semantic_type': 'ordinal'},
    {'variable': 'studytime', 'meaning': 'Thời gian tự học hàng tuần', 'storage_dtype': 'int64', 'semantic_type': 'ordinal'},
    {'variable': 'failures', 'meaning': 'Số lần rớt môn', 'storage_dtype': 'int64', 'semantic_type': 'discrete numerical'},
    {'variable': 'schoolsup', 'meaning': 'Hỗ trợ học tập từ trường', 'storage_dtype': 'object', 'semantic_type': 'binary'},
    {'variable': 'internet', 'meaning': 'Internet tại nhà', 'storage_dtype': 'object', 'semantic_type': 'binary'},
    {'variable': 'absences', 'meaning': 'Số ngày vắng mặt', 'storage_dtype': 'int64', 'semantic_type': 'discrete numerical'},
    {'variable': 'G1', 'meaning': 'Điểm kỳ 1', 'storage_dtype': 'int64', 'semantic_type': 'discrete numerical'},
    {'variable': 'G3', 'meaning': 'Điểm cuối kỳ', 'storage_dtype': 'int64', 'semantic_type': 'discrete numerical'}
])
display(data_dict)


## 3. Data Quality Audit
Tạo bảng gồm `dtype`, `missing`, `missing_pct`, `unique`. Kiểm tra duplicate và thống kê mô tả.

In [ ]:
# TODO 3: Data Quality Audit
audit = pd.DataFrame({
    'dtype': df.dtypes,
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().sum() / len(df)) * 100,
    'unique': df.nunique()
})
display(audit)
print('\nDuplicates:', df.duplicated().sum())
display(df.describe().T)


**Viết 5 nhận xét dựa trên bằng chứng:**
1. Tập dữ liệu gốc hoàn toàn không có giá trị bị khuyết (missing = 0) ở tất cả các cột.
2. Không có dòng dữ liệu nào bị trùng lặp hoàn toàn (Duplicates = 0).
3. Biến mục tiêu G3 có giá trị dao động từ 0 đến 19, với giá trị trung bình khoảng 11.9.
4. Cột absences (số ngày vắng mặt) có mức trung bình là 3.65 nhưng giá trị cực đại lên tới 32, cho thấy dấu hiệu của các outliers lệch phải.
5. Hầu hết học sinh không rớt môn nào trong quá khứ (cột failures có phân vị thứ 75 (Q3) là 0).

## 4. EDA — Phân phối `G3`

In [ ]:
# TODO 4: Vẽ histogram G3
plt.figure(figsize=(8, 5))
plt.hist(df['G3'], bins=20, edgecolor='black', color='skyblue')
plt.title('Phân phối điểm số cuối kỳ (G3)')
plt.xlabel('Điểm số G3')
plt.ylabel('Số lượng học sinh')
plt.show()


**Nhận xét:** Phân phối điểm G3 có dạng gần giống hình chuông (chuẩn) với đỉnh tập trung ở khoảng 11-12 điểm. Tuy nhiên, có một lượng nhỏ học sinh đạt mức điểm 0 bất thường, có thể là do bỏ thi hoặc không nộp bài cuối kỳ.

## 5. EDA — `absences` và outlier trực quan

In [ ]:
# TODO 5: Boxplot absences + describe()
plt.figure(figsize=(8, 4))
plt.boxplot(df['absences'], vert=False)
plt.title('Boxplot phân phối số ngày vắng mặt (absences)')
plt.xlabel('Số ngày vắng')
plt.show()

display(df['absences'].describe())


**Trả lời:** Có nên tự động xóa mọi điểm bất thường? Vì sao?  
**Không nên tự động xóa mọi điểm bất thường.** Các giá trị ngoại lệ (ví dụ: vắng mặt > 20 ngày) rất có thể là dữ liệu thực tế, phản ánh những học sinh có vấn đề về sức khỏe, hoàn cảnh hoặc nguy cơ bỏ học cao. Việc xóa bỏ chúng sẽ làm mất thông tin quan trọng giúp nhà trường dự đoán học sinh có điểm G3 thấp.

## 6. So sánh các nhóm

In [ ]:
print("So sánh theo Internet:")
display(df.groupby('internet')['G3'].agg(['count', 'mean', 'median', 'std']))
print("\nSo sánh theo Mong muốn học lên cao (higher):")
display(df.groupby('higher')['G3'].agg(['count', 'mean', 'median', 'std']))
print("\nSo sánh theo Thời gian tự học (studytime):")
display(df.groupby('studytime')['G3'].agg(['count', 'mean', 'median', 'std']))
print("\nSo sánh theo Số lần rớt môn (failures):")
display(df.groupby('failures')['G3'].agg(['count', 'mean', 'median', 'std']))


**Câu hỏi:** Nếu `internet=yes` có G3 trung bình cao hơn, có thể kết luận Internet *gây ra* điểm cao hơn không?  
**Không.** Mối tương quan (correlation) không đồng nghĩa với quan hệ nhân quả (causation). Học sinh có internet=yes đạt điểm trung bình cao hơn có thể do các yếu tố ẩn khác chi phối (ví dụ: điều kiện kinh tế gia đình tốt hơn, có môi trường học tập đầy đủ hơn), chứ không nhất thiết vì bản thân cái Internet trực tiếp làm điểm cao lên.

## 7. Correlation Analysis

In [ ]:
num_cols = [
    'age','Medu','Fedu','studytime','failures','famrel','freetime',
    'goout','Dalc','Walc','health','absences','G1','G2','G3'
]

# TODO 7: tính correlation matrix và sắp xếp correlation với G3
corr_matrix = df[num_cols].corr()
print("Mức độ tương quan với G3 (Sắp xếp giảm dần):")
display(corr_matrix['G3'].sort_values(ascending=False))


**Nhận xét về G1/G2 và nguy cơ leakage theo prediction time:** `G1` và `G2` có mức độ tương quan cực kỳ cao với `G3` (> 0.8). Nếu mục tiêu của mô hình là cảnh báo sớm nguy cơ rớt môn **từ đầu năm học**, việc sử dụng `G1` và `G2` sẽ gây ra **Data Leakage** (rò rỉ dữ liệu) vì hai biến này chứa thông tin từ tương lai mà ở thời điểm đầu năm học ta chưa thể biết được.

## 8. Tạo dirty dataset có kiểm soát
Chạy cell sau nguyên trạng để mọi nhóm có cùng lỗi dữ liệu.

In [ ]:
dirty = df.copy()
rng = np.random.default_rng(42)

# Missing ở studytime
idx = rng.choice(dirty.index, size=20, replace=False)
dirty.loc[idx, 'studytime'] = np.nan

# Missing ở Mjob
idx = rng.choice(dirty.index, size=15, replace=False)
dirty.loc[idx, 'Mjob'] = np.nan

# Category không nhất quán
idx = rng.choice(dirty.index, size=8, replace=False)
dirty.loc[idx, 'school'] = 'gp'

# Duplicate 5 dòng đầu
dirty = pd.concat([dirty, dirty.iloc[:5]], ignore_index=True)

# Extreme value có chủ đích
dirty.loc[0, 'absences'] = 250

print('Dirty shape:', dirty.shape)


## 9. Data Cleaning

In [ ]:
# TODO 9A: chuẩn hóa school về uppercase
dirty['school'] = dirty['school'].str.upper()

# TODO 9B: phát hiện và drop duplicates; báo cáo số dòng trước/sau
print("Số dòng trước khi xóa duplicate:", dirty.shape[0])
dirty = dirty.drop_duplicates(keep='first').reset_index(drop=True)
print("Số dòng sau khi xóa duplicate:", dirty.shape[0])

# TODO 9C: fill missing studytime bằng median
dirty['studytime'] = dirty['studytime'].fillna(dirty['studytime'].median())

# TODO 9D: fill missing Mjob bằng mode
dirty['Mjob'] = dirty['Mjob'].fillna(dirty['Mjob'].mode()[0])

# TODO 9E: chạy lại audit để xác nhận
audit_dirty = pd.DataFrame({
    'missing_sau_khi_fill': dirty.isna().sum(),
    'unique': dirty.nunique()
})
display(audit_dirty[audit_dirty['missing_sau_khi_fill'] > 0])


**Giải thích:** Vì sao không nên `fillna(df.mean())` cho toàn bộ cột?  
Không nên dùng `fillna(df.mean())` cho toàn bộ các cột vì các biến Categorical/Nominal (như `Mjob`) không có ý nghĩa toán học để tính trung bình. Ngoài ra, với các biến số bị lệch hoặc chứa giá trị outlier quá cao, giá trị trung bình (mean) sẽ bị kéo lệch đáng kể; sử dụng trung vị (median) hoặc mode sẽ an toàn hơn.

## 10. Outlier Detection bằng IQR

In [ ]:
# TODO 10: tính Q1, Q3, IQR, lower, upper cho absences
Q1 = dirty['absences'].quantile(0.25)
Q3 = dirty['absences'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = dirty[(dirty['absences'] < lower_bound) | (dirty['absences'] > upper_bound)]
print(f"Khoảng IQR hợp lệ: [{lower_bound}, {upper_bound}]")
print(f"Số lượng outliers tìm thấy: {len(outliers)}")
display(outliers[['absences', 'G3']])


**Phân loại các trường hợp đáng chú ý:**  
- **Data Error:** Học sinh vắng 250 ngày là vô lý trong 1 năm học (cần loại bỏ hoặc sửa lại).
- **Potential Real Extreme Value:** Vắng từ khoảng 12 đến 32 ngày là những con số cao nhưng có thể hoàn toàn là sự thật (đau ốm, trốn học nhiều).
- **Need Domain Verification:** Cần đem các số liệu như vắng > 20 ngày đi xác minh lại với hồ sơ điểm danh của giáo viên để quyết định xử lý (giữ lại hoặc capping).

## 11. One-hot Encoding

In [ ]:
nominal_cols = ['school','sex','address','famsize','Pstatus','Mjob','Fjob','reason','guardian']

# TODO 11: one-hot encoding từ dirty sau cleaning
print("Shape trước khi mã hóa:", dirty.shape)
dirty_encoded = pd.get_dummies(dirty, columns=nominal_cols, drop_first=True)
print("Shape sau khi mã hóa:", dirty_encoded.shape)


## 12. Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

features_scale = ['age','studytime','failures','absences']

# TODO 12: StandardScaler -> scaled_df
scaler = StandardScaler()
scaled_df = dirty_encoded.copy()

scaled_df[features_scale] = scaler.fit_transform(scaled_df[features_scale])
display(scaled_df[features_scale].describe().round(3))


**Câu hỏi:** Nếu không scale, feature nào có thể chi phối Euclidean distance?  
Nếu không scale, tính năng `absences` (có dải giá trị từ 0 tới hàng chục) sẽ làm lu mờ hoàn toàn sự chênh lệch của các tính năng như `failures` hoặc `studytime` (chỉ từ 1-4) trong việc tính toán khoảng cách. Do đó `absences` sẽ chi phối toàn bộ thước đo khoảng cách Euclidean.

## 13. Similarity / Distance

In [ ]:
from scipy.spatial.distance import euclidean, cityblock, cosine

# TODO 13: tính Euclidean, Manhattan, Cosine distance giữa hai học viên đầu tiên
v0 = scaled_df[features_scale].iloc[0].values
v1 = scaled_df[features_scale].iloc[1].values

print(f"Euclidean distance: {euclidean(v0, v1):.4f}")
print(f"Manhattan (Cityblock) distance: {cityblock(v0, v1):.4f}")
print(f"Cosine distance: {cosine(v0, v1):.4f}")


**Diễn giải:** Khoảng cách Euclidean đo độ dài đường thẳng giữa hai điểm dữ liệu. Manhattan đo khoảng cách theo đường lưới trực giao (tổng độ chênh lệch tuyệt đối từng chiều). Cosine đo độ lệch về góc (hướng) giữa hai vector học viên, bỏ qua quy mô.

## 14. Tìm 5 học viên gần nhất

In [ ]:
from sklearn.metrics import pairwise_distances

# TODO 14: pairwise Euclidean distances trên scaled_df
query_id = 0
dist_matrix = pairwise_distances(scaled_df[features_scale], metric='euclidean')

nearest_indices = np.argsort(dist_matrix[query_id])[1:6]

print("5 học sinh có profile tương đồng nhất với học sinh ở index 0:")
display(dirty.iloc[nearest_indices][features_scale + ['G1','G2','G3']])


**Kết luận ngắn:** Học viên có profile gần nhau có nhất thiết có G3 giống nhau không?  
**Không.** Việc có profile tương đồng (tuổi, số giờ tự học, vắng mặt) không nhất thiết đảm bảo các học sinh sẽ có điểm G3 giống hệt nhau. Điểm số phụ thuộc vào nhiều yếu tố chưa được đo lường (thông minh, sức khỏe tâm lý, cách giảng dạy của giáo viên, v.v.).

## 15. Data Leakage theo Prediction Time
Điền bảng sau bằng Có/Không và giải thích.

| Scenario | Prediction time | Dùng G1? | Dùng G2? | Giải thích |
|---|---|---|---|---|
| A | Đầu năm | Không | Không | Cả kỳ 1 và kỳ 2 đều chưa diễn ra. Việc dùng điểm này dự báo đầu năm là lấy thông tin tương lai. |
| B | Sau kỳ 1 | Có | Không | Điểm kỳ 1 (G1) đã có thật. Nhưng kỳ 2 (G2) thì chưa kết thúc, dùng G2 lúc này là Data Leakage. |
| C | Sau kỳ 2 | Có | Có | Đã hoàn thành 2 học kỳ, cả 2 cột điểm G1 và G2 đều hợp lệ để dùng làm đầu vào dự báo. |

## 16. Tạo Mining-Ready Dataset cho early warning đầu học kỳ

In [ ]:
# TODO 16A: y = 1 nếu G3 < 10, else 0
y = (dirty_encoded['G3'] < 10).astype(int).rename('Target_Fail')

# TODO 16B: X bỏ G1, G2, G3
X = dirty_encoded.drop(columns=['G1', 'G2', 'G3'])

# TODO 16C: lưu mining_ready_early_warning.csv
mining_df = pd.concat([X, y], axis=1)
mining_df.to_csv('mining_ready_early_warning.csv', index=False)
print("Đã xuất file dữ liệu 'mining_ready_early_warning.csv'. Shape:", mining_df.shape)


## 17. Kết luận 150–250 từ
Quá trình phân tích tập dữ liệu "Student Performance" (1) đã chỉ ra một điều nổi bật là thành tích học tập bị chi phối bởi rất nhiều yếu tố ngoại cảnh nhưng tương quan mạnh nhất với kết quả trước đó của chính học sinh (điểm G1, G2). (2) Quá trình làm sạch đã phát hiện các vấn đề chất lượng như: dữ liệu bị trùng lặp, thiếu giá trị ngẫu nhiên và sự xuất hiện của các giá trị cực đoan phi thực tế ở số ngày nghỉ (`absences` = 250). (3) Dù vậy, một giới hạn của quá trình EDA (chỉ so sánh 2 chiều) là không phát hiện được sự tương tác đa biến phức tạp giữa các yếu tố với nhau. (4) Trong mô hình dự báo, nguy cơ lớn nhất là "Data Leakage" nếu vô tình dùng điểm G1 và G2 để dự báo sớm đầu năm học, hoặc đưa trực tiếp điểm số G3 vào dữ liệu học (train data) gây ra dự đoán hoàn hảo ảo. (5) Đến hiện tại, file `mining_ready_early_warning.csv` đã hoàn toàn sẵn sàng cho quá trình Train mô hình học máy: nó đã loại bỏ các biến leakage, xử lý missing value, làm phẳng dữ liệu danh mục (One-hot Encoding) và quy về mục tiêu nhị phân để phân loại trực tiếp nguy cơ rớt môn.

## 18. Checklist nộp bài
- [x] Notebook chạy từ đầu đến cuối
- [x] Có Data Quality Audit
- [x] Có ≥ 4 biểu đồ EDA và diễn giải
- [x] Có cleaning log
- [x] Có distance comparison
- [x] Có leakage analysis
- [x] Có `mining_ready_early_warning.csv`
- [x] Có kết luận 150–250 từ